# 01 — Simulation Engine (premium-independent)
Bread and butter. Generates the book (`gen`) and evolves it 5 years (`simulate`): frequency → peril → severity → retention → NCD. Output carries claims + labels, **no premium** — pricing happens in `02_pricing.ipynb`.

Contract: output columns follow `voltvision.config.COLS` (`POLID` first). Each scenario saves to `shared/sim_<SCEN>.parquet` (pickle fallback). Run top to bottom; only touch the config cell.

In [1]:
import copy, os, sys
ROOT = os.path.abspath('')
if ROOT not in sys.path: sys.path.insert(0, ROOT)

from voltvision import CFG, SCEN, N_YEARS, COLS, gen, simulate
from voltvision import io


## Config (only cell you need to touch)
| Knob | Default | Effect |
|---|---|---|
| `VEH` | `"ALL"` | `"ICE"` / `"EV"` / `"MIX"` single run, `"ALL"` loops all three |
| `QUICK` | `False` | `True` → n=1000, 2 years (smoke test) |
| `CFG` | `voltvision/config.py` | book size, horizon, frequency base (−2.00), EV severity 1.20, NCD table, tariff inputs |

In [2]:
VEH = "ALL"   # "ICE" | "EV" | "MIX" | "ALL"
QUICK = False  # True -> fast smoke run

cfg = copy.deepcopy(CFG)
if QUICK:
    cfg['n'] = 1000
scens = [VEH] if VEH in SCEN else ["ICE", "EV", "MIX"]
nyears = 2 if QUICK else N_YEARS
print(scens, {s: SCEN[s] for s in scens})
print('n =', cfg['n'], '| years =', nyears, '| seed =', cfg['seed'])


['ICE', 'EV', 'MIX'] {'ICE': {'ICE': 1.0, 'EV': 0.0}, 'EV': {'ICE': 0.0, 'EV': 1.0}, 'MIX': {'ICE': 0.9, 'EV': 0.1}}
n = 10000 | years = 5 | seed = 20260916


In [3]:
for s in scens:
    vp = SCEN[s]
    d0 = gen(cfg, vp, cfg['seed'])
    book = simulate(d0, cfg, vp, seed=cfg['seed'], n_years=nyears, verbose=True)
    p = io.save_sim(s, book)
    print(f"{s}: {len(book)} rows -> {p}\n")
io.save_config(cfg)
print('config snapshot -> shared/config.json')


Year 2026: 10000 pols, claims 1827, freq 16.2%
Year 2027: 13805 pols, claims 2517, freq 16.5%
Year 2028: 17180 pols, claims 3118, freq 16.4%
Year 2029: 20264 pols, claims 3546, freq 15.9%
Year 2030: 23029 pols, claims 4087, freq 16.1%
ICE: 84278 rows -> C:\Users\Admin\Documents\Visual Studio Code\MET-X\shared\sim_ICE.pkl

Year 2026: 10000 pols, claims 1930, freq 17.1%
Year 2027: 13752 pols, claims 2597, freq 17.1%
Year 2028: 17117 pols, claims 3236, freq 17.1%
Year 2029: 20136 pols, claims 3752, freq 16.8%
Year 2030: 22857 pols, claims 4310, freq 17.0%
EV: 83862 rows -> C:\Users\Admin\Documents\Visual Studio Code\MET-X\shared\sim_EV.pkl

Year 2026: 10000 pols, claims 1844, freq 16.4%
Year 2027: 13765 pols, claims 2486, freq 16.4%
Year 2028: 17151 pols, claims 3147, freq 16.6%
Year 2029: 20182 pols, claims 3630, freq 16.4%
Year 2030: 22943 pols, claims 4121, freq 16.3%
MIX: 84041 rows -> C:\Users\Admin\Documents\Visual Studio Code\MET-X\shared\sim_MIX.pkl

config snapshot -> shared/conf

## Notes / improvement backlog
- Flood/theft flag thresholds are complemented vs full-notebook `risk_pct` — flagged in `voltvision/simulate.py`, behavior kept until approved.
- Candidates: EV-ramp entrant mix, entrant NCD profile, vectorized NCD lookup.
- Next: `02_pricing.ipynb` prices these books three ways. Full run + figures live in `03_main.ipynb`.